# Regional identity signature calculation

This version normalizes multi-bin RSI with the training mean calculated separately for every parcel and every bin.

## Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path('..').resolve()
from rise.features import load_feature_config
from rise.rsi import (
    build_group_expression_matrix,
    build_group_reference_embeddings,
    calculate_adaptive_threshold,
    compute_region_corrs_parallel,
    compute_test_rsi,
    compute_training_rsi,
    load_split_embeddings,
)

model_name = 'rise_hcp_pretrain_models'
model_dir = project_root / 'models' / model_name
train_id_file = project_root / 'data/HCP/info/train.txt'
test_id_file = project_root / 'data/HCP/info/test.txt'
feature_config = load_feature_config(
    project_root / 'configs/hcp_feature_parameters.jsonc'
)
atlas_info_file = feature_config['atlas_mapping_file']
output_dir = project_root / 'results/rsi/HCP'
run_name = 'mean'

train_subject_ids = pd.read_csv(train_id_file, header=None, dtype=str)[0].tolist()
test_subject_ids = pd.read_csv(test_id_file, header=None, dtype=str)[0].tolist()
atlas_info = pd.read_csv(atlas_info_file)
sample_name_list = atlas_info['Parcel name'].astype(str).tolist()
if len(sample_name_list) != 7340:
    raise ValueError(f'Expected 7340 parcels, found {len(sample_name_list)}')

output_dir.mkdir(parents=True, exist_ok=True)

print('Model name:', model_name)
print('Model directory:', model_dir)
print('Output directory:', output_dir)
print('Train subjects:', len(train_subject_ids))
print('Test subjects:', len(test_subject_ids))
print('Atlas parcels:', len(sample_name_list))

## Load embeddings and build the training group reference

In [ ]:
loaded_train_subject_ids, train_data_all_sample_dict_list, repeat_names = load_split_embeddings(
    model_dir,
    run_name=run_name,
    split='train',
    subject_ids=train_subject_ids,
)
loaded_test_subject_ids, test_data_all_sample_dict_list, _ = load_split_embeddings(
    model_dir,
    run_name=run_name,
    split='test',
    repeat_names=repeat_names,
    subject_ids=test_subject_ids,
)
train_subject_ids = loaded_train_subject_ids
test_subject_ids = loaded_test_subject_ids
train_group_mean_lh_list, train_group_mean_rh_list = build_group_reference_embeddings(
    train_data_all_sample_dict_list
)

print('Repeats:', repeat_names)
print('Train subjects:', len(train_subject_ids))
print('Test subjects:', len(test_subject_ids))
print('Left group-reference shape:', train_group_mean_lh_list[0].shape)
print('Right group-reference shape:', train_group_mean_rh_list[0].shape)

## Calculate individual expression matrices and the adaptive threshold

Train and test expression matrices are calculated once with subject-level thread parallelism and retained for the RSI calculation.

In [ ]:
similarity = 'cosine'
n_jobs = 64

train_indiv_expression_matrices = compute_region_corrs_parallel(
    train_subject_ids,
    train_group_mean_lh_list,
    train_group_mean_rh_list,
    train_data_all_sample_dict_list,
    repeat_num=len(repeat_names),
    similarity=similarity,
    n_jobs=n_jobs,
)

In [ ]:
test_indiv_expression_matrices = compute_region_corrs_parallel(
    test_subject_ids,
    train_group_mean_lh_list,
    train_group_mean_rh_list,
    test_data_all_sample_dict_list,
    repeat_num=len(repeat_names),
    similarity=similarity,
    n_jobs=n_jobs,
)

In [ ]:
training_group_expression_matrix = build_group_expression_matrix(
    train_indiv_expression_matrices,
)
threshold_percentile = 99
adaptive_threshold = calculate_adaptive_threshold(
    training_group_expression_matrix,
    percentile=threshold_percentile,
)
print('Training group expression shape:', training_group_expression_matrix.shape)
print('Train individual expression shape:', train_indiv_expression_matrices.shape)
print('Test individual expression shape:', test_indiv_expression_matrices.shape)

del training_group_expression_matrix

In [ ]:
training_group_expression_matrix = build_group_expression_matrix(
    train_indiv_expression_matrices,
)

row_thresholds = np.percentile(
        training_group_expression_matrix, 99, axis=1
    )
threshold = float(np.mean(row_thresholds))
print(threshold)

## Calculate training and test RSI


In [ ]:
num_bins = 10

train_rsi, train_multibin_rsi, train_group_mean_rsi, train_multibin_group_mean_rsi = compute_training_rsi(
    train_indiv_expression_matrices,
    train_subject_ids,
    adaptive_threshold,
    num_bins=num_bins,
    n_jobs=n_jobs,
)


In [ ]:
test_rsi, test_multibin_rsi = compute_test_rsi(
    test_indiv_expression_matrices,
    test_subject_ids,
    adaptive_threshold,
    training_group_mean=train_group_mean_rsi,
    training_multibin_group_mean=train_multibin_group_mean_rsi,
    num_bins=num_bins,
    n_jobs=n_jobs,
)


print('Train single-bin RSI:', train_rsi.shape)
print('Train multi-bin RSI:', train_multibin_rsi.shape)
print('Test single-bin RSI:', test_rsi.shape)
print('Test multi-bin RSI:', test_multibin_rsi.shape)


## Save results

In [ ]:
# Existing single-bin filenames remain the raw/original RSI outputs.
train_rsi.to_csv(output_dir / 'train_rsi.csv')
test_rsi.to_csv(output_dir / 'test_rsi.csv')

# Multibin outputs are raw only.
np.save(output_dir / 'test_multibin_rsi.npy', test_multibin_rsi)